# 03 · Modelos · Random Forest y XGBoost

## ¿Qué vamos a resolver?

Queremos anticipar si un cliente contratará un depósito a plazo (`y = 1`) para ayudar al banco a priorizar sus llamadas. En este notebook comparamos dos modelos de árboles con los mismos datos procesados y el mismo preprocesado que usa el resto del equipo.

La clase positiva es minoritaria (aprox. 11 %). Nos interesa encontrar compradores (**recall**), pero también evitar demasiadas llamadas improductivas (**precisión**). El **F1** resume el equilibrio entre ambas: se usa para escoger los hiperparámetros, no para ocultar el compromiso entre recall y precisión.

## Recorrido del notebook

1. Cargar solo los datos de entrenamiento y fijar la validación común.
2. Comparar Random Forest y XGBoost con 4 configuraciones acotadas cada uno.
3. Repetir cada búsqueda con y sin SMOTE, siempre dentro del pipeline de cada fold.
4. Registrar las configuraciones en MLflow y comparar las mejores con las referencias del equipo.

**Para exponerlo:** primero explica por qué el desbalanceo importa; luego, cómo evitamos fuga de información con el pipeline y los folds; por último, presenta el intercambio entre recall y precisión y justifica la recomendación con F1.


## 0. Preparación: bibliotecas y utilidades

Importamos los dos clasificadores, `GridSearchCV`, SMOTE y las herramientas compartidas del proyecto. `src/utils.py` centraliza la carga de los CSV, el preprocesador común y el registro en MLflow; así evitamos que cada notebook implemente esas decisiones de forma distinta.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import make_scorer, precision_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from xgboost import XGBClassifier

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

from src.utils import (
    cargar_datos_procesados,
    crear_pipeline,
    crear_preprocesador,
    registrar_en_mlflow,
)


## 1. Datos y validación

La división train/test ya se hizo en el notebook de preparación. Aquí el ajuste y la comparación usan exclusivamente `X_train` e `y_train`; las salidas de test que devuelve la función común se descartan y no se evalúan.

Usamos cinco folds estratificados, con mezcla reproducible (`random_state=42`), para conservar la proporción de las clases en cada partición. En cada fold se calculan recall, precisión y F1 tanto en el subconjunto de ajuste como en el de validación. El tiempo mostrado es el promedio de ajuste de un fold.

- **Recall:** qué proporción de los clientes que contratarían logra detectar el modelo.
- **Precisión:** qué proporción de los clientes marcados para llamar contrataría realmente.
- **F1:** equilibrio entre recall y precisión. La búsqueda lo maximiza; después revisamos también las otras dos métricas antes de recomendar un modelo.

In [ ]:
SEMILLA = 42
N_FOLDS = 5
N_JOBS_CV = 2


## 2. Datos de entrenamiento y criterio común

La función `cargar_datos_procesados()` carga los CSV preparados por el equipo. La salida de esta celda confirma el tamaño de train y la tasa positiva que se mantendrá durante la comparación; no hay una nueva limpieza ni una nueva división.

In [1]:
# Se conservan solo train para los experimentos; las salidas de test se descartan.
X_train, _, y_train, _ = cargar_datos_procesados()
cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEMILLA)
scoring = {
    "recall": "recall",
    "precision": make_scorer(precision_score, zero_division=0),
    "f1": "f1",
}

print(f"X_train: {X_train.shape} | y_train: {y_train.shape}")
print(f"Proporción positiva en train: {y_train.mean():.3%}")
print(f"CV: {N_FOLDS} folds estratificados | selección de parámetros: F1 de validación")

X_train: (32940, 21) | y_train: (32940,)
Proporción positiva en train: 11.266%
CV: 5 folds estratificados | selección de parámetros: F1 de validación


## 3. Preprocesamiento y experimento con SMOTE

El preprocesador compartido aplica `StandardScaler` a las variables numéricas y `OneHotEncoder(handle_unknown="ignore")` a las categóricas. Aunque los árboles no necesitan escalar las variables, conservamos el pipeline común del proyecto. Al estar dentro del pipeline, el preprocesador se ajusta solo con el train de cada fold.

Sin SMOTE reutilizamos `crear_pipeline()`. Con SMOTE usamos `imblearn.pipeline.Pipeline`, en el orden **preprocesado → SMOTE → modelo**. Así el sobremuestreo ocurre únicamente durante el ajuste de cada fold: la validación conserva los datos originales y no se filtra información entre particiones.

**Limitación:** SMOTE después de OneHotEncoder puede interpolar columnas categóricas y crear valores fraccionarios que no corresponden a categorías reales. Aquí mantenemos ese diseño para que la comparación sea coherente con el experimento MLP del equipo; `SMOTENC` antes del one-hot sería una alternativa más respetuosa con las categorías, pero requeriría cambiar el preprocesado común.

In [ ]:
def crear_pipeline_experimento(modelo, usar_smote):
    """Construye el pipeline común y añade SMOTE solo cuando se solicita."""
    if usar_smote:
        return ImbPipeline([
            ("preprocesado", crear_preprocesador()),
            ("smote", SMOTE(random_state=SEMILLA)),
            ("modelo", modelo),
        ])
    return crear_pipeline(modelo)


def ejecutar_busqueda(nombre_modelo, nombre_base, modelo, grilla, usar_smote):
    """Busca con CV, registra cada candidato y devuelve el mejor por F1 medio."""
    busqueda = GridSearchCV(
        estimator=crear_pipeline_experimento(modelo, usar_smote),
        param_grid=grilla,
        scoring=scoring,
        refit="f1",
        cv=cv,
        n_jobs=N_JOBS_CV,
        pre_dispatch="n_jobs",
        return_train_score=True,
        error_score="raise",
    )
    busqueda.fit(X_train, y_train)

    resultados_cv = busqueda.cv_results_
    indice_mejor = busqueda.best_index_
    candidatos = []
    mejores_parametros = {
        clave.removeprefix("modelo__"): valor
        for clave, valor in busqueda.best_params_.items()
    }

    for indice, parametros_cv in enumerate(resultados_cv["params"]):
        es_mejor = indice == indice_mejor
        parametros_modelo = {
            clave.removeprefix("modelo__"): valor
            for clave, valor in parametros_cv.items()
        }
        metricas = {
            "recall_train": float(resultados_cv["mean_train_recall"][indice]),
            "recall_val": float(resultados_cv["mean_test_recall"][indice]),
            "precision_train": float(resultados_cv["mean_train_precision"][indice]),
            "precision_val": float(resultados_cv["mean_test_precision"][indice]),
            "f1_train": float(resultados_cv["mean_train_f1"][indice]),
            "f1_val": float(resultados_cv["mean_test_f1"][indice]),
            "tiempo_entrenamiento_s": float(resultados_cv["mean_fit_time"][indice]),
        }
        params_mlflow = {
            "modelo": nombre_modelo,
            "smote": usar_smote,
            "cv_folds": N_FOLDS,
            "criterio_seleccion": "f1_validacion",
            "configuracion": indice + 1,
            "seleccionada": es_mejor,
            "rango_f1": int(resultados_cv["rank_test_f1"][indice]),
            **parametros_modelo,
        }
        if es_mejor:
            params_mlflow["mejores_parametros"] = json.dumps(
                mejores_parametros, sort_keys=True
            )
        nombre_run = (
            nombre_base
            if es_mejor
            else f"{nombre_base}_candidato_{indice + 1:02d}"
        )
        registrar_en_mlflow(nombre_run, params_mlflow, metricas)
        candidatos.append({
            "modelo": nombre_modelo,
            "smote": usar_smote,
            "configuracion": indice + 1,
            "seleccionada": es_mejor,
            "parametros": parametros_modelo,
            **metricas,
        })

    mejor = {
        "modelo": nombre_modelo,
        "smote": usar_smote,
        "mejores_parametros": mejores_parametros,
        **candidatos[indice_mejor],
    }
    return mejor, candidatos


## 4. Random Forest

Random Forest construye muchos árboles sobre muestras y subconjuntos de variables distintos y combina sus predicciones. La combinación suele reducir la variabilidad de un árbol individual. No necesita escalado por sí mismo, pero usamos el preprocesado compartido para comparar los modelos bajo el mismo flujo.

### 4.1 Búsqueda de hiperparámetros

Probamos cuatro configuraciones diseñadas para explorar complejidad, regularización de las hojas y compensación de clases sin formar una búsqueda exhaustiva. La tabla que genera el código incluye cada combinación, sus métricas de train/validación y el tiempo medio; GridSearchCV selecciona la de mayor F1 de validación para cada condición de SMOTE.

| Hiperparámetro | Qué controla | Qué comparamos |
|---|---|---|
| `n_estimators` | Número de árboles | 200–300 árboles |
| `max_depth` | Profundidad máxima de cada árbol | Sin límite y límites 16/20 |
| `min_samples_split` / `min_samples_leaf` | Cuándo dividir y el tamaño mínimo de una hoja | Valores 2/5 y 1/2/5 |
| `max_features` | Variables candidatas en cada división | `sqrt` y 0,8 |
| `class_weight` | Peso relativo de cada clase | Sin ponderar y `balanced` |

Se usa una semilla fija y un hilo por bosque. La búsqueda paraleliza como máximo dos ajustes para evitar multiplicar innecesariamente los hilos.

In [ ]:
grilla_rf = [
    {
        "modelo__n_estimators": [200], "modelo__max_depth": [None],
        "modelo__min_samples_split": [2], "modelo__min_samples_leaf": [1],
        "modelo__max_features": ["sqrt"], "modelo__class_weight": [None],
    },
    {
        "modelo__n_estimators": [250], "modelo__max_depth": [16],
        "modelo__min_samples_split": [5], "modelo__min_samples_leaf": [2],
        "modelo__max_features": ["sqrt"], "modelo__class_weight": [None],
    },
    {
        "modelo__n_estimators": [300], "modelo__max_depth": [20],
        "modelo__min_samples_split": [2], "modelo__min_samples_leaf": [2],
        "modelo__max_features": [0.8], "modelo__class_weight": [None],
    },
    {
        "modelo__n_estimators": [250], "modelo__max_depth": [16],
        "modelo__min_samples_split": [2], "modelo__min_samples_leaf": [5],
        "modelo__max_features": ["sqrt"], "modelo__class_weight": ["balanced"],
    },
]


### 4.2 Ajuste y comparación con/sin SMOTE

La misma cuadrícula se evalúa primero con los datos originales y luego con SMOTE dentro del pipeline. Cada candidato se registra en MLflow; el mejor de cada búsqueda recibe el nombre claro `random_forest_sin_smote` o `random_forest_smote`. La selección es por F1, no por el máximo recall aislado.

In [2]:
resultados_finales = []
candidatos_rf = []
for usar_smote in (False, True):
    nombre_base = f"random_forest_{'smote' if usar_smote else 'sin_smote'}"
    mejor, candidatos = ejecutar_busqueda(
        nombre_modelo="RandomForestClassifier",
        nombre_base=nombre_base,
        modelo=RandomForestClassifier(random_state=SEMILLA, n_jobs=1),
        grilla=grilla_rf,
        usar_smote=usar_smote,
    )
    resultados_finales.append(mejor)
    candidatos_rf.extend(candidatos)
    print(
        f"{nombre_base}: F1-val={mejor['f1_val']:.3f}, "
        f"recall-val={mejor['recall_val']:.3f}, "
        f"precision-val={mejor['precision_val']:.3f}"
    )

df_candidatos_rf = pd.DataFrame(candidatos_rf).sort_values(
    ["f1_val", "recall_val"], ascending=False
).reset_index(drop=True)
display(df_candidatos_rf)


2026/11/12 20:34:45 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/11/12 20:34:45 INFO mlflow.store.db.utils: Updating database tables


2026/11/12 20:34:48 INFO mlflow.tracking.fluent: Experiment with name 'reto1-deposito-banco' does not exist. Creating a new experiment.


random_forest_sin_smote: F1-val=0.483, recall-val=0.610, precision-val=0.401


random_forest_smote: F1-val=0.490, recall-val=0.507, precision-val=0.474


,modelo,smote,configuracion,seleccionada,parametros,recall_train,recall_val,precision_train,precision_val,f1_train,f1_val,tiempo_entrenamiento_s
0,RandomForestClassifier,True,4,True,"{'class_weight': 'balanced', 'max_depth': 16, ...",0.586837,0.507406,0.565306,0.474009,0.575819,0.490098,12.631728
1,RandomForestClassifier,True,2,False,"{'class_weight': None, 'max_depth': 16, 'max_f...",0.601995,0.484502,0.640947,0.484374,0.620824,0.484386,12.233826
2,RandomForestClassifier,False,4,True,"{'class_weight': 'balanced', 'max_depth': 16, ...",0.713219,0.609808,0.451175,0.400525,0.552692,0.483466,4.390152
3,RandomForestClassifier,True,3,False,"{'class_weight': None, 'max_depth': 20, 'max_f...",0.684856,0.367556,0.911155,0.519214,0.781904,0.430327,80.462566
4,RandomForestClassifier,True,1,False,"{'class_weight': None, 'max_depth': None, 'max...",0.974198,0.350038,0.988049,0.505386,0.981073,0.413520,10.838264
5,RandomForestClassifier,False,3,False,"{'class_weight': None, 'max_depth': 20, 'max_f...",0.588184,0.283215,0.982228,0.593656,0.735733,0.383337,28.631467
6,RandomForestClassifier,False,1,False,"{'class_weight': None, 'max_depth': None, 'max...",0.967057,0.290223,0.995423,0.553228,0.981035,0.380544,4.112937
7,RandomForestClassifier,False,2,False,"{'class_weight': None, 'max_depth': 16, 'max_f...",0.413904,0.262737,0.916046,0.645660,0.570162,0.373309,4.413117


### 4.3 Lectura de Random Forest

En la mejor configuración sin SMOTE, el recall de validación es **0,610**, la precisión **0,401** y el F1 **0,483**. Con SMOTE, la misma configuración seleccionada obtiene recall **0,507**, precisión **0,474** y F1 **0,490**. En este caso SMOTE mejora ligeramente el F1 y la precisión, pero baja el recall: detecta menos compradores, aunque las llamadas seleccionadas son algo más precisas.

La diferencia F1 train-validación es aproximadamente **0,069** sin SMOTE y **0,086** con SMOTE; hay una brecha que conviene vigilar. La tabla de candidatos permite ver si la diferencia se debe también a hiperparámetros distintos o a la condición de sobremuestreo.

## 5. XGBoost

XGBoost añade árboles de forma secuencial: cada nuevo árbol intenta corregir errores de los anteriores. Es un método de *gradient boosting*, distinto del entrenamiento en paralelo de los árboles de Random Forest. Se configura para clasificación binaria (`objective="binary:logistic"`), usa `logloss` como función interna de evaluación y `tree_method="hist"` para ajustar con histogramas.

### 5.1 Búsqueda de hiperparámetros

Se comparan cuatro combinaciones acotadas. `n_estimators`, `max_depth` y `learning_rate` controlan el número, la complejidad y el tamaño de paso del boosting; `subsample` y `colsample_bytree` añaden aleatoriedad, mientras que `min_child_weight` y `reg_lambda` ayudan a regularizar. Cada modelo usa un hilo y GridSearchCV un máximo de dos ajustes en paralelo.

In [ ]:
grilla_xgb = [
    {
        "modelo__n_estimators": [200], "modelo__max_depth": [3],
        "modelo__learning_rate": [0.05], "modelo__subsample": [0.8],
        "modelo__colsample_bytree": [0.8], "modelo__min_child_weight": [1],
        "modelo__reg_lambda": [1.0],
    },
    {
        "modelo__n_estimators": [300], "modelo__max_depth": [3],
        "modelo__learning_rate": [0.03], "modelo__subsample": [0.8],
        "modelo__colsample_bytree": [0.8], "modelo__min_child_weight": [1],
        "modelo__reg_lambda": [5.0],
    },
    {
        "modelo__n_estimators": [200], "modelo__max_depth": [5],
        "modelo__learning_rate": [0.05], "modelo__subsample": [0.8],
        "modelo__colsample_bytree": [0.8], "modelo__min_child_weight": [3],
        "modelo__reg_lambda": [3.0],
    },
    {
        "modelo__n_estimators": [300], "modelo__max_depth": [5],
        "modelo__learning_rate": [0.03], "modelo__subsample": [0.9],
        "modelo__colsample_bytree": [0.8], "modelo__min_child_weight": [3],
        "modelo__reg_lambda": [5.0],
    },
]


### 5.2 Ajuste y comparación con/sin SMOTE

Se conserva la configuración binaria y la semilla. Al igual que en Random Forest, la única diferencia entre las dos búsquedas es incluir o no SMOTE en el pipeline; la validación nunca se sobremuestrea. Las mejores configuraciones se registran como `xgboost_sin_smote` y `xgboost_smote`, junto con todos sus candidatos.

In [3]:
candidatos_xgb = []
for usar_smote in (False, True):
    nombre_base = f"xgboost_{'smote' if usar_smote else 'sin_smote'}"
    mejor, candidatos = ejecutar_busqueda(
        nombre_modelo="XGBClassifier",
        nombre_base=nombre_base,
        modelo=XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            random_state=SEMILLA,
            n_jobs=1,
            verbosity=0,
        ),
        grilla=grilla_xgb,
        usar_smote=usar_smote,
    )
    resultados_finales.append(mejor)
    candidatos_xgb.extend(candidatos)
    print(
        f"{nombre_base}: F1-val={mejor['f1_val']:.3f}, "
        f"recall-val={mejor['recall_val']:.3f}, "
        f"precision-val={mejor['precision_val']:.3f}"
    )

df_candidatos_xgb = pd.DataFrame(candidatos_xgb).sort_values(
    ["f1_val", "recall_val"], ascending=False
).reset_index(drop=True)
display(df_candidatos_xgb)


xgboost_sin_smote: F1-val=0.371, recall-val=0.259, precision-val=0.652


xgboost_smote: F1-val=0.471, recall-val=0.469, precision-val=0.474


,modelo,smote,configuracion,seleccionada,parametros,recall_train,recall_val,precision_train,precision_val,f1_train,f1_val,tiempo_entrenamiento_s
0,XGBClassifier,True,2,True,"{'colsample_bytree': 0.8, 'learning_rate': 0.0...",0.480396,0.468871,0.485197,0.473534,0.482774,0.471104,4.936766
1,XGBClassifier,True,1,False,"{'colsample_bytree': 0.8, 'learning_rate': 0.0...",0.467057,0.453511,0.497108,0.486146,0.481555,0.469205,2.697955
2,XGBClassifier,True,4,False,"{'colsample_bytree': 0.8, 'learning_rate': 0.0...",0.432161,0.389109,0.576499,0.528592,0.493859,0.448248,5.206759
3,XGBClassifier,True,3,False,"{'colsample_bytree': 0.8, 'learning_rate': 0.0...",0.417811,0.369709,0.600193,0.537334,0.492423,0.437987,4.879886
4,XGBClassifier,False,4,True,"{'colsample_bytree': 0.8, 'learning_rate': 0.0...",0.298841,0.259499,0.740668,0.651713,0.425833,0.371061,1.895626
5,XGBClassifier,False,3,False,"{'colsample_bytree': 0.8, 'learning_rate': 0.0...",0.309283,0.260307,0.751081,0.644574,0.438103,0.370665,1.334615
6,XGBClassifier,False,1,False,"{'colsample_bytree': 0.8, 'learning_rate': 0.0...",0.258892,0.245754,0.713342,0.673641,0.379884,0.360036,1.236637
7,XGBClassifier,False,2,False,"{'colsample_bytree': 0.8, 'learning_rate': 0.0...",0.248518,0.238478,0.705987,0.678841,0.367594,0.352895,1.570331


### 5.3 Lectura de XGBoost

Sin SMOTE, XGBoost consigue **0,259 de recall**, **0,652 de precisión** y **0,371 de F1** en validación: marca menos clientes, pero una mayor proporción de los marcados contrataría. Con SMOTE, el recall sube a **0,469** y el F1 a **0,471**, mientras que la precisión baja a **0,474**. Es un intercambio más marcado entre ventas potenciales detectadas y llamadas improductivas.

La brecha F1 train-validación es de aproximadamente **0,055** sin SMOTE y **0,012** con SMOTE. En estas mejores configuraciones, el sobremuestreo de XGBoost acompaña una brecha menor y un recall sustancialmente mayor; esto no significa que SMOTE siempre mejore todos los modelos.

## 6. Comparación de las mejores configuraciones

La tabla final contiene una fila por búsqueda (modelo × SMOTE), con los mejores hiperparámetros y las medias de train/validación de los cinco folds. Se ordena por F1 de validación, usando recall y precisión como criterios secundarios. No es una evaluación con test.

Como referencias previas del README, **no recalculadas aquí**: regresión logística (recall 0,231; precisión 0,664; F1 0,343), KNN (0,298; 0,481; 0,368), SVM lineal (0,622; 0,352; 0,450) y MLP regularizado con SMOTE (F1 0,388).

In [4]:
columnas_tabla = [
    "modelo", "smote", "mejores_parametros",
    "recall_train", "recall_val", "precision_train", "precision_val",
    "f1_train", "f1_val", "tiempo_entrenamiento_s",
]
df_comparacion = pd.DataFrame(resultados_finales)
df_comparacion = df_comparacion.sort_values(
    ["f1_val", "recall_val", "precision_val"], ascending=False
).reset_index(drop=True)
display(df_comparacion[columnas_tabla].style.format({
    "recall_train": "{:.3f}", "recall_val": "{:.3f}",
    "precision_train": "{:.3f}", "precision_val": "{:.3f}",
    "f1_train": "{:.3f}", "f1_val": "{:.3f}",
    "tiempo_entrenamiento_s": "{:.2f}",
}))

mejor_recall = df_comparacion.loc[df_comparacion["recall_val"].idxmax()]
mejor_precision = df_comparacion.loc[df_comparacion["precision_val"].idxmax()]
mejor_f1 = df_comparacion.loc[df_comparacion["f1_val"].idxmax()]
print(
    f"Mejor recall de validación: {mejor_recall['modelo']} "
    f"(SMOTE={mejor_recall['smote']}), {mejor_recall['recall_val']:.3f}."
)
print(
    f"Mejor precisión de validación: {mejor_precision['modelo']} "
    f"(SMOTE={mejor_precision['smote']}), {mejor_precision['precision_val']:.3f}."
)
print(
    f"Mejor F1 de validación: {mejor_f1['modelo']} "
    f"(SMOTE={mejor_f1['smote']}), {mejor_f1['f1_val']:.3f}."
)

print("\nCambio entre las mejores búsquedas con y sin SMOTE:")
for nombre_modelo in df_comparacion["modelo"].drop_duplicates():
    grupo = df_comparacion[df_comparacion["modelo"] == nombre_modelo].set_index("smote")
    sin_smote = grupo.loc[False]
    con_smote = grupo.loc[True]
    cambios = {
        metrica: con_smote[f"{metrica}_val"] - sin_smote[f"{metrica}_val"]
        for metrica in ("recall", "precision", "f1")
    }
    resumen = ", ".join(f"{metrica} {cambio:+.3f}" for metrica, cambio in cambios.items())
    print(f"{nombre_modelo}: {resumen} (con SMOTE menos sin SMOTE).")

brechas = df_comparacion.assign(
    brecha_f1=lambda tabla: tabla["f1_train"] - tabla["f1_val"]
)
mayor_brecha = brechas.loc[brechas["brecha_f1"].idxmax()]
if mayor_brecha["brecha_f1"] > 0:
    print(
        f"\nHay indicio de sobreajuste en {mayor_brecha['modelo']} "
        f"(SMOTE={mayor_brecha['smote']}): la brecha F1 train-validación "
        f"más grande es {mayor_brecha['brecha_f1']:.3f}. Comprueba también "
        "las otras brechas en la tabla."
    )
else:
    print("\nNo aparece una brecha positiva F1 train-validación en estas cuatro mejores configuraciones.")

print(
    f"\nConfiguración recomendada para la comparación global: {mejor_f1['modelo']} "
    f"(SMOTE={mejor_f1['smote']}), por su mayor F1 de validación "
    f"({mejor_f1['f1_val']:.3f}); su recall es {mejor_f1['recall_val']:.3f} "
    f"y su precisión {mejor_f1['precision_val']:.3f}. No es una evaluación final "
    "con test: esa comparación se reserva para el flujo final del proyecto."
)

,modelo,smote,mejores_parametros,recall_train,recall_val,precision_train,precision_val,f1_train,f1_val,tiempo_entrenamiento_s
0,RandomForestClassifier,True,"{'class_weight': 'balanced', 'max_depth': 16, 'max_features': 'sqrt', 'min_samples_leaf': 5, 'min_samples_split': 2, 'n_estimators': 250}",0.587,0.507,0.565,0.474,0.576,0.490,12.63
1,RandomForestClassifier,False,"{'class_weight': 'balanced', 'max_depth': 16, 'max_features': 'sqrt', 'min_samples_leaf': 5, 'min_samples_split': 2, 'n_estimators': 250}",0.713,0.610,0.451,0.401,0.553,0.483,4.39
2,XGBClassifier,True,"{'colsample_bytree': 0.8, 'learning_rate': 0.03, 'max_depth': 3, 'min_child_weight': 1, 'n_estimators': 300, 'reg_lambda': 5.0, 'subsample': 0.8}",0.480,0.469,0.485,0.474,0.483,0.471,4.94
3,XGBClassifier,False,"{'colsample_bytree': 0.8, 'learning_rate': 0.03, 'max_depth': 5, 'min_child_weight': 3, 'n_estimators': 300, 'reg_lambda': 5.0, 'subsample': 0.9}",0.299,0.259,0.741,0.652,0.426,0.371,1.90


Mejor recall de validación: RandomForestClassifier (SMOTE=False), 0.610.
Mejor precisión de validación: XGBClassifier (SMOTE=False), 0.652.
Mejor F1 de validación: RandomForestClassifier (SMOTE=True), 0.490.

Cambio entre las mejores búsquedas con y sin SMOTE:
RandomForestClassifier: recall -0.102, precision +0.073, f1 +0.007 (con SMOTE menos sin SMOTE).
XGBClassifier: recall +0.209, precision -0.178, f1 +0.100 (con SMOTE menos sin SMOTE).

Hay indicio de sobreajuste en RandomForestClassifier (SMOTE=True): la brecha F1 train-validación más grande es 0.086. Comprueba también las otras brechas en la tabla.

Configuración recomendada para la comparación global: RandomForestClassifier (SMOTE=True), por su mayor F1 de validación (0.490); su recall es 0.507 y su precisión 0.474. No es una evaluación final con test: esa comparación se reserva para el flujo final del proyecto.


## 7. Conclusiones para la decisión de negocio

- **Mayor recall entre estos experimentos:** Random Forest sin SMOTE (**0,610**). Es la opción que detecta más de los clientes positivos de los dos modelos, aunque su precisión es **0,401**.
- **Mayor precisión:** XGBoost sin SMOTE (**0,652**), a cambio de un recall de **0,259**.
- **Mayor F1:** Random Forest con SMOTE (**0,490**). Lo recomendaría como candidato para la comparación global porque equilibra mejor las dos métricas en estos experimentos; si la prioridad fuera estrictamente detectar el máximo de compradores, elegiríamos Random Forest sin SMOTE y aceptaríamos su menor precisión.
- **Efecto de SMOTE:** en Random Forest el F1 apenas aumenta (+0,007) y el recall disminuye (−0,102); en XGBoost suben recall (+0,209) y F1 (+0,100), pero la precisión baja (−0,178). El efecto depende del modelo y exige valorar el coste de las llamadas.
- **Sobreajuste:** la mayor brecha F1 train-validación entre las cuatro mejores configuraciones es **0,086** (Random Forest con SMOTE). Conviene tenerla presente y no interpretar la media de validación como resultado final.

Como contexto, el mejor F1 de estos experimentos (**0,490**) supera las referencias del SVM (**0,450**) y del MLP regularizado (**0,388**). El mayor recall (**0,610**) queda algo por debajo del SVM (**0,622**) y la mayor precisión (**0,652**) algo por debajo de la regresión logística (**0,664**): no hay un ganador en todas las métricas.

La recomendación es para comparar en el proyecto, no para desplegar todavía. El conjunto de test se mantiene reservado para la evaluación final del modelo elegido en el flujo previsto por el equipo.

## 8. Registro y reproducción en MLflow

Se registraron **16 ejecuciones** en el experimento común `reto1-deposito-banco`: 4 candidatos por cada combinación de modelo y uso de SMOTE. Cada ejecución guarda sus hiperparámetros, la condición SMOTE, los 5 folds, las métricas de train/validación y el tiempo medio de ajuste. Las ejecuciones principales se llaman `random_forest_sin_smote`, `random_forest_smote`, `xgboost_sin_smote` y `xgboost_smote`; las demás identifican el candidato.

Para consultarlas desde la raíz del proyecto: `mlflow ui --backend-store-uri sqlite:///mlflow.db`.